In [2]:
import pandas
df = pd.read_csv('Detailed_Polling_Data.csv')

<IPython.core.display.Javascript object>

In [3]:
df.columns

Index(['serial no. of polling station',
       'all india anna dravida munnetra kazhagam', 'naam tamilar katchi',
       'dravida munnetra kazhagam', 'tamizhaga vaazhvurimai katchi',
       'naam indiar party', 'aanaithinthiya jananayaka pathukappu kazhagam',
       'bahujan dravida party', 'puthiya makkal tamil desam katchi',
       'tamilaga vettri kazhagam', 'thamizhaka padaippalar makkal katchi',
       'puthiya tamilagam', 'independent', 'independent.1', 'independent.2',
       'independent.3', 'independent.4', 'independent.5', 'independent.6',
       'independent.7', 'independent.8', 'total of valid votes',
       'no. of rejected votes', 'nota', 'total', 'no. of tendered votes',
       'building in which it will be located', 'polling areas',
       'whether for all voters or men only or women only', 'building_clean',
       'location_clean', 'Winner_Party', 'Winner_Votes', 'Runner_Up_Votes',
       'Margin_Of_Victory', 'Runner_Up_Party', 'AIADMK_Rank', 'NTK_Rank',
       'DMK_Ra

In [4]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 1. Load the new dataset (Update the filename to match your file)
df7 = pd.read_csv("Detailed_Polling_Data.csv")

# 2. Select the primary political party columns for analysis
core_parties = [
    'all india anna dravida munnetra kazhagam', 
    'dravida munnetra kazhagam', 
    'naam tamilar katchi', 
    'tamilaga vettri kazhagam'
]

# Clean missing numerical fields by filling with 0
df7[core_parties] = df7[core_parties].fillna(0)

# 3. Calculate true total votes for normalization (Core + Independents + NOTA)
df7['Total_Calculated_Votes'] = df7[core_parties].sum(axis=1) + df7['Total_Independent_Votes'].fillna(0) + df7['nota'].fillna(0)

# Filter out empty entries to completely avoid division by zero errors
df7 = df7[df7['Total_Calculated_Votes'] > 0].copy()

# 4. Feature Engineering: Create normalized percentage shares (%)
share_cols = []
for party in core_parties:
    col_name = f'{party}_share_pct'
    df7[col_name] = (df7[party] / df7['Total_Calculated_Votes']) * 100
    share_cols.append(col_name)

# Append strategic structural dimensions
df7['independent_share_pct'] = (df7['Total_Independent_Votes'].fillna(0) / df7['Total_Calculated_Votes']) * 100
feature_cols = share_cols + ['independent_share_pct', 'Margin_Percentage']

# Drop or fill edge-case missing numbers inside target features
df7[feature_cols] = df7[feature_cols].fillna(0)

# 5. Extract and Scale features for the ML model
X = df7[feature_cols]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 6. Apply K-Means Clustering to group booths into 4 core segments
optimal_k = 4
kmeans = KMeans(n_clusters=optimal_k, init='k-means++', random_state=42, n_init=10)
df7['Cluster_ID'] = kmeans.fit_predict(X_scaled)

# 7. Print the Raw Profile Breakdown to help map the text identities
print("\n--- DATASET 7: AVERAGE VOTE SHARE & METRICS PER CLUSTER ---")
profile = df7.groupby('Cluster_ID')[feature_cols].mean()
print(profile.round(2))

print("\n--- DATASET 7: BOOTH COUNT PER CLUSTER ---")
print(df7['Cluster_ID'].value_counts())

# 8. Export individual target files for campaign ground teams
for cluster_num in range(optimal_k):
    cluster_df = df7[df7['Cluster_ID'] == cluster_num][
        [
            'serial no. of polling station', 
            'location_clean', 
            'building_clean', 
            'polling areas', 
            'Winner_Party', 
            'Margin_Percentage'
        ]
    ]
    filename = f"Dataset_7_Cluster_{cluster_num}_Booths.csv"
    cluster_df.to_csv(filename, index=False)

print("\nSuccess! Campaign target files generated for all 4 clusters.")



--- DATASET 7: AVERAGE VOTE SHARE & METRICS PER CLUSTER ---
            all india anna dravida munnetra kazhagam_share_pct  \
Cluster_ID                                                       
0                                                       20.72    
1                                                        8.50    
2                                                       44.77    
3                                                       22.91    

            dravida munnetra kazhagam_share_pct  \
Cluster_ID                                        
0                                         31.24   
1                                         63.42   
2                                         17.53   
3                                         28.43   

            naam tamilar katchi_share_pct  tamilaga vettri kazhagam_share_pct  \
Cluster_ID                                                                      
0                                   10.97                               3